# SQL Gym — 03: Joins

Practice: `INNER JOIN`, `LEFT JOIN`, anti-joins, self-joins, and multi-table join chains.
Write your SQL in the `%%solution N` cell and run it — results preview inline. Then run the check cell to validate.

**Tables:** `users`, `accounts`, `merchants`, `transactions`

In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd()
_candidates = [_cwd / "sql", _cwd, _cwd.parent, _cwd.parent / "sql",
               _cwd.parent.parent, _cwd.parent.parent / "sql"]
_sql_dir = next((p for p in _candidates if (p / "utils" / "__init__.py").exists()), None)
if _sql_dir is None:
    raise RuntimeError(
        "Cannot locate sql/utils. Run: uv run jupyter lab from the project root."
    )
if str(_sql_dir) not in sys.path:
    sys.path.insert(0, str(_sql_dir))

DATA_DIR = _sql_dir / "data"

from utils import get_conn, check, register_sql_magic
from utils.checks.joins import Checker

conn = get_conn(DATA_DIR)
checker = Checker(conn)
register_sql_magic()
print("Ready. Tables: users, merchants, accounts, transactions")

Ready. Tables: users, merchants, accounts, transactions


In [2]:
for table in ["users", "merchants", "accounts", "transactions"]:
    print(f"\n{'─'*50}\n  {table}\n{'─'*50}")
    display(conn.execute(f"SELECT * FROM {table} LIMIT 3").df())
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  ({n:,} rows total)")


──────────────────────────────────────────────────
  users
──────────────────────────────────────────────────


,user_id,name,email,country,tier,kyc_verified,created_date
0,1,Carlos Clark,user1@example.com,IN,basic,True,2021-02-09
1,2,Tara Hernandez,user2@example.com,SG,basic,False,2022-12-11
2,3,Marcus Robinson,user3@example.com,AU,basic,True,2020-02-15


  (500 rows total)

──────────────────────────────────────────────────
  merchants
──────────────────────────────────────────────────


,merchant_id,name,mcc_category,city,country
0,1,DailyBasket Groceries 1,groceries,Austin,UK
1,2,GreenLeaf Groceries 2,groceries,New York,IN
2,3,DailyBasket Groceries 3,groceries,Chicago,UK


  (200 rows total)

──────────────────────────────────────────────────
  accounts
──────────────────────────────────────────────────


,account_id,user_id,account_type,opened_date,balance,status
0,1,1,checking,2021-02-21,2365.36,closed
1,2,2,savings,2023-05-16,-1171.39,active
2,3,3,credit,2020-04-07,22128.47,closed


  (575 rows total)

──────────────────────────────────────────────────
  transactions
──────────────────────────────────────────────────


,txn_id,account_id,merchant_id,amount,txn_type,txn_date,status
0,1,463,69,3621.72,credit,2024-09-22,completed
1,2,498,157,175.48,credit,2024-01-11,completed
2,3,49,181,383.48,debit,2024-04-12,completed


  (20,000 rows total)


## Problem 1: Full Transaction Details

Enrich every transaction with user info (name, country, tier), account type, and merchant info (name, category). Return all transactions, ordered by txn_id.

<details>
<summary>Hint</summary>

Chain four INNER JOINs: `transactions → accounts` (on account_id) `→ users` (on user_id) `→ merchants` (on merchant_id). Use `u.name AS user_name` and `m.name AS merchant_name` to disambiguate. `USING (col)` works when column names match; `ON t.account_id = a.account_id` is more explicit — interviewers appreciate you knowing both.

</details>

| Column | Type | Notes |
|--------|------|-------|
| txn_id | integer | |
| txn_date | date | |
| amount | double | |
| txn_type | string | |
| status | string | |
| user_name | string | user's full name |
| country | string | |
| tier | string | |
| account_type | string | |
| merchant_name | string | merchant's name |
| mcc_category | string | |

Expected: all **20,000 transactions**, ordered by txn_id.

In [3]:
%%solution 1

SELECT
txn_id,
txn_date,
amount,
txn_type,
txn.status,
users.name AS user_name,
users.country,
tier,
account_type,
mr.name AS merchant_name,
mcc_category
FROM transactions txn
JOIN merchants mr ON txn.merchant_id = mr.merchant_id
JOIN accounts acc ON txn.account_id = acc.account_id
JOIN users ON acc.user_id = users.user_id
ORDER BY txn_id

,txn_id,txn_date,amount,txn_type,status,user_name,country,tier,account_type,merchant_name,mcc_category
0,1,2024-09-22,3621.72,credit,completed,Hannah Hall,US,basic,savings,SkyWay Travel 19,travel
1,2,2024-01-11,175.48,credit,completed,Marcus Moore,US,basic,credit,MediCare Healthcare 7,healthcare
2,3,2024-04-12,383.48,debit,completed,Hannah Lee,IN,business,savings,FuelStop Fuel 6,fuel
3,4,2023-07-22,2537.60,debit,completed,William Garcia,US,basic,checking,UrbanEats Restaurants 9,restaurants
4,5,2024-02-18,34.06,debit,completed,Samuel Anderson,IN,basic,checking,StyleHub Retail 16,retail
...,...,...,...,...,...,...,...,...,...,...,...
19995,19996,2024-02-25,114.17,credit,completed,Chloe Wilson,US,premium,checking,GreenLeaf Groceries 23,groceries
19996,19997,2023-02-10,14.09,debit,completed,Tara Brown,UK,basic,savings,LiveBeat Entertainment 25,entertainment
19997,19998,2022-10-18,360.04,debit,completed,Hannah Lee,IN,business,savings,RoamEasy Travel 15,travel
19998,19999,2023-04-17,302.49,credit,completed,Carlos Martinez,AU,premium,credit,TrendStore Retail 9,retail


In [4]:
checker.p1(solution_1)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 2: Users Who Have Never Made a Transaction

Find users with no transactions whatsoever (not even failed or pending). Return their profile details sorted by user_id.

<details>
<summary>Hint</summary>

Anti-join pattern. LEFT JOIN `users → accounts → transactions`, then filter `WHERE t.txn_id IS NULL`. This is the standard anti-join idiom across all SQL dialects. Alternatively: `WHERE user_id NOT IN (SELECT DISTINCT a.user_id FROM accounts a JOIN transactions t ON a.account_id = t.account_id)`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| user_id | integer | |
| name | string | |
| email | string | |
| country | string | |
| tier | string | |

Expected: variable rows, ordered by user_id ASC.

In [9]:
%%solution 2

SELECT
users.user_id,
name,
email,
country,
tier
FROM users
LEFT JOIN accounts acc ON users.user_id = acc.user_id
LEFT JOIN transactions txn ON acc.account_id = txn.account_id
WHERE txn.txn_id IS NULL


,user_id,name,email,country,tier
0,476,Carlos Martinez,user476@example.com,IN,premium
1,477,Samuel Thomas,user477@example.com,UK,basic
2,478,Xiu White,user478@example.com,UK,business
3,479,Marcus Lopez,user479@example.com,UK,basic
4,480,Dev Thomas,user480@example.com,US,basic
5,481,Ethan Thomas,user481@example.com,AU,basic
6,482,Zoe Miller,user482@example.com,SG,basic
7,483,Chloe Thompson,user483@example.com,IN,premium
8,484,Samuel Martinez,user484@example.com,UK,basic
9,485,Bram Johnson,user485@example.com,US,premium


In [10]:
checker.p2(solution_2)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 3: Revenue by MCC Category for Premium/Business Users

Among premium and business tier users only, compute transaction count and total revenue for each MCC category (completed transactions only). Sort by mcc_category, then tier.

<details>
<summary>Hint</summary>

Four-table join. Filter `u.tier IN ('premium', 'business')` and `t.status = 'completed'`. Group by `mcc_category`, `tier`. This is a filtered aggregation via join — no subquery needed.

</details>

| Column | Type | Notes |
|--------|------|-------|
| mcc_category | string | |
| tier | string | |
| txn_count | bigint | |
| total_revenue | double | 2 decimal places |

Expected: up to **16 rows** (8 categories × 2 tiers), ordered by mcc_category ASC, tier ASC.

In [24]:
%%solution 3

SELECT
mcc_category,
tier,
COUNT(*) as txn_count,
ROUND(SUM(amount), 2) as total_revenue
FROM transactions txn
JOIN accounts acc ON acc.account_id = txn.account_id
JOIN users ON acc.user_id = users.user_id
JOIN merchants mr ON txn.merchant_id = mr.merchant_id
WHERE txn.status = 'completed' AND tier IN ('premium', 'business')
GROUP BY 1, 2
ORDER BY 1, 2

,mcc_category,tier,txn_count,total_revenue
0,entertainment,business,469,80159.55
1,entertainment,premium,418,60033.78
2,fuel,business,491,93904.86
3,fuel,premium,471,71050.76
4,groceries,business,465,74188.74
5,groceries,premium,437,63897.27
6,healthcare,business,428,103113.53
7,healthcare,premium,435,64927.66
8,restaurants,business,440,75585.87
9,restaurants,premium,432,74414.67


In [25]:
checker.p3(solution_3)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 4: Accounts with Both Debit and Credit Transactions

Find accounts that have at least one completed debit AND at least one completed credit transaction. Return account details sorted by account_id.

<details>
<summary>Hint</summary>

Use subquery intersection: `WHERE account_id IN (SELECT ... WHERE txn_type = 'debit') AND account_id IN (SELECT ... WHERE txn_type = 'credit')`. Alternative approach (self-join): `JOIN transactions d ON ... JOIN transactions c ON d.account_id = c.account_id AND d.txn_type = 'debit' AND c.txn_type = 'credit'` — remember `DISTINCT` to avoid row explosion.

</details>

| Column | Type | Notes |
|--------|------|-------|
| account_id | integer | |
| account_type | string | |
| user_id | integer | |

Expected: variable rows, ordered by account_id ASC.

In [40]:
%%solution 4

WITH credit_txn_acc_users AS (
    SELECT
    acc.account_id,
    account_type,
    users.user_id
    FROM transactions txn
    JOIN accounts acc ON txn.account_id = acc.account_id
    JOIN users ON acc.user_id = users.user_id
    WHERE txn.txn_type = 'credit' AND txn.status = 'completed'
    GROUP BY 1, 2, 3
), debit_txn_acc_users AS (
    SELECT
    acc.account_id,
    account_type,
    users.user_id
    FROM transactions txn
    JOIN accounts acc ON txn.account_id = acc.account_id
    JOIN users ON acc.user_id = users.user_id
    WHERE txn.txn_type = 'debit' AND txn.status = 'completed'
    GROUP BY 1, 2, 3
)
SELECT cu.*
FROM credit_txn_acc_users cu
JOIN debit_txn_acc_users du ON cu.account_id = du.account_id
ORDER BY cu.account_id

,account_id,account_type,user_id
0,1,checking,1
1,2,savings,2
2,3,credit,3
3,4,checking,4
4,5,credit,5
...,...,...,...
567,571,checking,68
568,572,checking,214
569,573,savings,441
570,574,credit,296


In [41]:
checker.p4(solution_4)  # type: ignore[name-defined]  # noqa: F821

True

## Problem 5: Users Whose Spend Exceeds Their Country Average

For completed debit transactions, find users whose total spend is above the average total spend for their country. Show each user's total spend alongside their country's average.

<details>
<summary>Hint</summary>

Two CTEs: first compute per-user total spend, then compute per-country average of those totals. JOIN them on `country` and filter `WHERE total_spend > country_avg_spend`. This pattern (aggregate → re-aggregate → compare) is very common in analytics interviews.

</details>

| Column | Type | Notes |
|--------|------|-------|
| user_id | integer | |
| name | string | |
| country | string | |
| total_spend | double | 2 decimal places |
| country_avg_spend | double | 2 decimal places, average for the user's country |

Expected: variable rows, ordered by country ASC, total_spend DESC.

In [ ]:
%%solution 5

In [ ]:
checker.p5(solution_5)  # type: ignore[name-defined]  # noqa: F821